# SpottingSalmon
YOLO model logging script
- To register the final model as an MLFlow model


In [0]:
%pip install --quiet numpy==1.26.4  mlflow ultralytics

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
import mlflow.pyfunc
import ultralytics
from mlflow.models.signature import ModelSignature
from mlflow.types import Schema, TensorSpec, DataType, ColSpec
import numpy as np
from ultralytics import YOLO
from pyspark.sql import functions as F

### 1. Model logging

In [0]:
# Run ID and artifact relative path
run_id = "cc1ca4e121884869ac75efcba1a0af4a"
artifact_path = "weights/best.pt"

#model_path = f"runs:/{run_id}/{artifact_path}"
# Download to local temp dir
model_path = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path=artifact_path
)

print(f"Model downloaded to: {model_path}")

# Load the model
model = YOLO(model_path)

In [0]:
import mlflow.pyfunc
import pandas as pd
from ultralytics import YOLO
import cv2
import os  # for extracting video filename

def read_video_frames(video_path):
    """Generator that yields frames from a video file as numpy arrays."""
    cap = cv2.VideoCapture(video_path)
    frame_number = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        yield frame_number, frame
        frame_number += 1
    cap.release()

def convert_yolo_to_dets(results):
    """Convert YOLO Results objects to a list of dicts with bbox and confidence."""
    detections = []
    for r in results:  # results is a list of Results objects
        boxes = r.boxes
        if boxes is not None and len(boxes) > 0:
            xyxy = boxes.xyxy.cpu().numpy()   # (num_boxes, 4)
            confs = boxes.conf.cpu().numpy()  # (num_boxes,)
            for i in range(len(xyxy)):
                x1, y1, x2, y2 = xyxy[i]
                conf = float(confs[i])
                detections.append({
                    "x1": float(x1),
                    "y1": float(y1),
                    "x2": float(x2),
                    "y2": float(y2),
                    "confidence": conf
                })
    return detections

class FishVideoDetector(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        # Load YOLO checkpoint
        self.model = YOLO(context.artifacts["checkpoint"])

    def predict(self, context, model_input: pd.DataFrame):
        """
        model_input: pd.DataFrame with column 'fish' containing video file paths
        Output: pd.DataFrame with columns:
            fish_id, video, frame, x1, y1, x2, y2, confidence
        """
        all_detections = []

        for video_path in model_input["fish"]:
            video_name = os.path.basename(video_path)  # extract file name
            for frame_number, frame in read_video_frames(video_path):
                results = self.model(frame)  # YOLO detections
                dets = convert_yolo_to_dets(results)
                for det in dets:
                    det_row = {
                        "fish_id": None,          # placeholder for future tracking
                        "video": video_name,
                        "frame": frame_number,
                        **det
                    }
                    all_detections.append(det_row)

        return pd.DataFrame(all_detections)



In [0]:
import pandas as pd
from mlflow.models.signature import ModelSignature
from mlflow.types import Schema, ColSpec

input_example = pd.DataFrame({
    "fish": ["/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/4_2021-07-06_14-11-37x.mp4"]
})


signature = ModelSignature(
    inputs=Schema([ColSpec("string", "fish")]),
    outputs=Schema([
        ColSpec("integer", "fish_id"),
        ColSpec("string", "video"),
        ColSpec("integer", "frame"),
        ColSpec("float", "x1"),
        ColSpec("float", "y1"),
        ColSpec("float", "x2"),
        ColSpec("float", "y2"),
        ColSpec("float", "confidence")
    ])
)

In [0]:
with mlflow.start_run(run_id="cc1ca4e121884869ac75efcba1a0af4a"):

    mlflow.pyfunc.log_model(
        name="redolent-stag-178",
        python_model=FishVideoDetector(),   
        input_example=input_example,       
        artifacts={"checkpoint": model_path},
        signature=signature
    )

### Register model

In [0]:
mlflow.set_registry_uri("databricks-uc")

In [0]:
mlflow.register_model(
    model_uri="runs:/cc1ca4e121884869ac75efcba1a0af4a/redolent-stag-178",
    name="dev_dash_lab.alpha_restricted.salmon_model"
)

In [0]:
mlflow.register_model(
    model_uri="runs:/cc1ca4e121884869ac75efcba1a0af4a/redolent-stag-178",
    name="prd_dash_lab.dash_data_science_unrestricted.salmon_model_test"
)

### Test registered model 

In [0]:
model = mlflow.pyfunc.load_model("models:/prd_dash_lab.dash_data_science_unrestricted.salmon_model_test/9")

In [0]:
import pandas as pd
import os

# Directory where images are stored
image_dir = "/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/"

# List all files (filter for jpg/png)
image_files = [f for f in os.listdir(image_dir) if f.endswith((".mp4"))]

# Build full paths
full_paths = [os.path.join(image_dir, f) for f in image_files]

# Create a DataFrame suitable for your model
input_df = pd.DataFrame({
    "fish": full_paths
})

print(input_df.head())


In [0]:
results = model.predict(input_df)


In [0]:
display(results)

### Register model to workspace

In [0]:
mlflow.set_registry_uri("databricks")

mlflow.register_model(
    model_uri="runs:/cc1ca4e121884869ac75efcba1a0af4a/redolent-stag-178",
    name="FishVideoDetector"
)
